# **“정류장 500m 격자 피처 실험 결과 분석”**

이 노트북은 완료된 실험의 저장 예측만 읽어 결과표를 계산합니다. **학습·재학습 코드는 포함하지 않습니다.**

비교 기준은 전체 혼합 모델이나 ExtraTrees가 아니라 **평일 전용 LightGBM(`wd_lgb`, 36피처)**입니다. 이를 500m 격자 피처 두 개를 추가한 38피처 모델과 날짜별 검증 OOF 예측으로 비교합니다.

기존 구현에서 평일 전처리, 36개 피처, LightGBM 설정과 검증 구조를 활용했고, 이번 실험에서 좌표 격자 생성과 두 범주형 피처, 격자 모델 예측 저장·평가를 추가했습니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# 저장된 예측만 읽습니다. 이 노트북은 모델 학습을 실행하지 않습니다.
EXP_DIR = Path.cwd() / "dayun" / "grid_feature_experiment"
if not (EXP_DIR / "oof_grid_predictions.parquet").exists():
    EXP_DIR = Path.cwd()
    if EXP_DIR.name != "grid_feature_experiment":
        raise FileNotFoundError("프로젝트 루트 또는 grid_feature_experiment 폴더에서 실행하세요.")
oof = pd.read_parquet(EXP_DIR / "oof_grid_predictions.parquet")
assert len(oof) == 120_769 and oof["row_id"].is_unique

def mae(y, p):
    y, p = np.asarray(y, dtype=np.float64), np.asarray(p, dtype=np.float64)
    return float(np.mean(np.abs(y-p)))

def rmse(y, p):
    y, p = np.asarray(y, dtype=np.float64), np.asarray(p, dtype=np.float64)
    return float(np.sqrt(np.mean((y-p)**2)))

def score_table(frame, keys):
    rows = []
    for key, part in frame.groupby(keys, sort=True):
        if not isinstance(key, tuple): key = (key,)
        row = dict(zip(keys, key))
        y, base, grid = part.y_true, part.pred_base, part.pred_grid
        row.update(rows=len(part), MAE_base=mae(y,base), MAE_grid=mae(y,grid),
                   dMAE=mae(y,grid)-mae(y,base), RMSE_base=rmse(y,base),
                   RMSE_grid=rmse(y,grid), dRMSE=rmse(y,grid)-rmse(y,base))
        rows.append(row)
    return pd.DataFrame(rows)

y, base, grid = oof.y_true, oof.pred_base, oof.pred_grid
summary = pd.DataFrame([
    {"model":"평일 전용 wd_lgb 기준 (36피처)", "rows":len(oof), "Pooled_MAE":mae(y,base), "Pooled_RMSE":rmse(y,base)},
    {"model":"격자 추가 평일 LightGBM (38피처)", "rows":len(oof), "Pooled_MAE":mae(y,grid), "Pooled_RMSE":rmse(y,grid)},
])
daily = score_table(oof, ["date"])
route = score_table(oof, ["route_nm"])
date_route = score_table(oof, ["date", "route_nm"])
print("비교 기준: 전체 혼합 모델이 아닌 평일 전용 LightGBM(wd_lgb) 자체")
display(summary, daily, route, date_route)

## 저장 예측에서 다시 계산한 결과

전체 MAE는 평일 전용 기준 23.266초, 격자 추가 모델 23.274초로 +0.008초입니다. RMSE는 41.239초에서 41.324초로 +0.084초입니다. 전체적으로 소폭 악화됐습니다. 날짜별·노선별 표도 같은 저장 예측에서 계산합니다.

검증 행은 월요일 40,085행, 화요일 40,223행, 수요일 40,461행입니다. 결과표는 `summary_metrics.csv`, `daily_metrics.csv`, `route_metrics.csv`, `date_route_metrics.csv`와 대조할 수 있습니다.